In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold
import os
import random

from transformers import (
    AutoTokenizer
)

from data.datasets.Smiles_Dataset import SMILESDataset
from data.preprocessing.pipeline import Pipeline
from models.BERT import ChemBERTaClassifier
from utils.paths import WEIGHTS_DIR

/home/smirnov@dohod.local/Desktop/thortex-hackaton/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [3]:
from rdkit import Chem

def augment_rare_classes(df, rare_classes, augment_factor=5):
    augmented = []
    for _, row in df.iterrows():
        if row["taste_cluster"] in rare_classes:
            for _ in range(augment_factor):
                mol = Chem.MolFromSmiles(row["SMILES"])
                if mol is not None:
                    new_smiles = Chem.MolToSmiles(mol, doRandom=True)
                    augmented.append({
                        "SMILES": new_smiles,
                        "taste_cluster": row["taste_cluster"],
                    })
    return pd.DataFrame(augmented)

In [4]:
random_state = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 50
batch_size=64
lr=1e-4
dropout=0.1
val_size=0.25
patience=10
max_length=128
checkpoint_path="best_chemberta.pth"
save_dir_path = os.path.join(WEIGHTS_DIR, 'MLP')

set_seed(random_state)

In [5]:
pipeline = Pipeline(feature_type="morgan", download=False)

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

Extracting raw features: 100%|██████████| 888/888 [00:02<00:00, 401.68it/s]


In [6]:
def train_model(
    train_df,
    val_df,
    epochs=20,
    batch_size=64,
    lr=2e-5,
    dropout=0.3,
    patience=5,
    model_name="DeepChem/ChemBERTa-77M-MTR",
    max_length=128,
    checkpoint_path="best_chemberta.pth",
    save_dir_path=".",
    label_names=None
):
    set_seed()
    os.makedirs(save_dir_path, exist_ok=True)

    num_labels = train_df["taste_cluster"].nunique()

    if label_names is None:
        label_names = [str(i) for i in range(num_labels)]
    else:
        assert len(label_names) == num_labels, "label_names length must match number of classes"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = SMILESDataset(train_df, tokenizer, max_length=max_length, mode="train")
    val_dataset = SMILESDataset(val_df, tokenizer, max_length=max_length, mode="train")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = ChemBERTaClassifier(num_labels=num_labels, model_name=model_name, dropout=dropout).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(logits, labels)
                val_losses.append(loss.item())

                preds = torch.argmax(logits, dim=1)
                all_preds.append(preds.cpu().numpy())
                all_targets.append(labels.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

        val_preds = np.concatenate(all_preds)
        val_targets = np.concatenate(all_targets)
        macro_f1 = f1_score(val_targets, val_preds, average='macro')

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), os.path.join(save_dir_path, checkpoint_path))
            patience_counter = 0
            print(f"  → New best model saved! f1={best_f1:.5f}")
        else:
            patience_counter += 1
            print(f"  (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print("\nEarly stopping triggered!")
                break

    # -------------------------
    # Final evaluation
    # -------------------------
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    print("\nFinal macro F1 on validation:", macro_f1)

    # -------------------------
    # Classification report
    # -------------------------
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT (Validation Set)")
    print("=" * 60)
    report = classification_report(
        all_targets,
        all_preds,
        target_names=label_names,
        zero_division=0
    )
    print(report)

    return model, macro_f1, best_f1

In [7]:
def predict(model, test_df, batch_size=64, model_name="DeepChem/ChemBERTa-77M-MTR", max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    test_dataset = SMILESDataset(test_df, tokenizer, max_length=max_length, mode="test")
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_ids = []
    all_preds = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            ids = batch["id"]

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_ids.extend(ids)
            all_preds.extend(preds.cpu().numpy())

    pred_df = pd.DataFrame({"Pubchem_ID": all_ids, "taste_cluster": all_preds})
    return pred_df

In [8]:
train_df, test_df = pipeline.get_raw_data()

train_df = train_df.copy()
train_df["taste_cluster"] = train_df["taste_cluster"].astype(int)

train_df_orig, val_df = train_test_split(
    train_df,
    test_size=val_size,
    stratify=train_df["taste_cluster"],
    random_state=42
)

aug_df = augment_rare_classes(train_df_orig, rare_classes=[5, 6], augment_factor=8)
aug_df2 = augment_rare_classes(train_df_orig, rare_classes=[7, 2], augment_factor=3)
aug_df3 = augment_rare_classes(train_df_orig, rare_classes=[1, 3, 4], augment_factor=2)

train_df_balanced = pd.concat([train_df_orig, aug_df, aug_df2, aug_df3], ignore_index=True)
train_df_balanced["taste_cluster"] = train_df_balanced["taste_cluster"].astype(int)

model, f1, best_loss = train_model(
    train_df=train_df_balanced,
    val_df=val_df,
    epochs=epochs,
    batch_size=batch_size,
    lr=lr,
    dropout=dropout,
    patience=patience,
    max_length=max_length,
    checkpoint_path=checkpoint_path,
    save_dir_path=save_dir_path,
)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50 | train_loss=1.85625 | val_loss=1.75295
  → New best model saved! f1=0.26012
Epoch 2/50 | train_loss=1.43015 | val_loss=1.65328
  → New best model saved! f1=0.40135
Epoch 3/50 | train_loss=1.31098 | val_loss=1.62136
  → New best model saved! f1=0.41199
Epoch 4/50 | train_loss=1.23428 | val_loss=1.58943
  → New best model saved! f1=0.42868
Epoch 5/50 | train_loss=1.19614 | val_loss=1.60023
  → New best model saved! f1=0.43173
Epoch 6/50 | train_loss=1.13040 | val_loss=1.59613
  → New best model saved! f1=0.44766
Epoch 7/50 | train_loss=1.09781 | val_loss=1.58251
  → New best model saved! f1=0.45606
Epoch 8/50 | train_loss=1.06838 | val_loss=1.58663
  (patience 1/10)
Epoch 9/50 | train_loss=1.02889 | val_loss=1.53995
  → New best model saved! f1=0.46541
Epoch 10/50 | train_loss=0.99948 | val_loss=1.58433
  (patience 1/10)
Epoch 11/50 | train_loss=0.97028 | val_loss=1.61276
  (patience 2/10)
Epoch 12/50 | train_loss=0.95037 | val_loss=1.58663
  → New best model saved! f1=0.4828

In [9]:
model.load_state_dict(torch.load(os.path.join(save_dir_path, checkpoint_path)))
pred = predict(model, test_df)

In [11]:
pred['taste_cluster'].to_csv("submission_1.csv", index=False)